# Chapter 6: Replication
*From: Designing Data-Intensive Applications*

## TL;DR

- Replication keeps the **same data on multiple machines** to gain high availability, disconnected operation, low latency, durability, and read scalability — everything hard about it comes from handling *changes* on top of an unreliable network.
- There are exactly **three replication architectures** in practice: **single-leader** (one writer, fans out a log to followers), **multi-leader** (several writers, asynchronous cross-replication), and **leaderless** (clients write to many replicas in parallel and use quorums).
- Every approach chooses where it sits on the **synchronous ↔ asynchronous** axis. Sync gives durability but stalls on a slow replica; async gives availability and throughput but can silently lose recently-acknowledged writes on failover.
- Asynchronous replication exposes three user-visible anomalies — **read-your-writes**, **monotonic reads**, **consistent prefix reads** — that must be mitigated per-user before reaching for full linearizability.
- Concurrent writes (multi-leader or leaderless) require **conflict resolution**: avoidance, LWW, siblings/manual merge, or CRDT/OT. Detecting *whether* two writes are concurrent requires **version vectors**, not wall-clock timestamps.

## The Problem Replication Solves

- **High availability & disconnected operation**: keep serving reads and writes even if a machine, zone, or entire region goes down — or if the client itself is offline.
- **Latency via geo-proximity**: place a replica near each user so reads don't pay a cross-continent round trip; asynchronous writes can be absorbed locally.
- **Read scalability**: a read-heavy workload can fan out across many followers to exceed what a single machine could serve.

**What replication does NOT solve: point-in-time restore.** A deletion or a bad `UPDATE` is faithfully replicated to every follower in milliseconds. You still need **backups** (periodic snapshots + archived replication log) to travel backwards in time — replication and backups are complementary, not substitutes.

## Back-of-Envelope Estimation

The chapter's quantitative core is the **quorum condition** (`w + r > n`), the cost of **replication lag**, and the **lost-write window** of async cross-region replication. We compute all three.

In [ ]:
# =====================================================================
# 1. Quorum math: when does w + r > n actually guarantee an overlap?
# =====================================================================

def quorum_ok(n: int, w: int, r: int) -> dict:
    """Check the Dynamo-style quorum condition and failure tolerance."""
    overlap_guarantee = (w + r) > n
    write_failures_tolerated = n - w   # nodes that can be down and writes still succeed
    read_failures_tolerated  = n - r   # nodes that can be down and reads still succeed
    return {
        "n": n, "w": w, "r": r,
        "w+r": w + r,
        "overlap_guaranteed": overlap_guarantee,
        "write_failures_tolerated": write_failures_tolerated,
        "read_failures_tolerated":  read_failures_tolerated,
    }

configs = [
    quorum_ok(3, 2, 2),   # classic Dynamo default
    quorum_ok(5, 3, 3),   # tolerates 2 failures
    quorum_ok(5, 1, 5),   # fast writes, slow reads, no failure tolerance on reads
    quorum_ok(3, 1, 1),   # fails quorum: stale reads possible
]
for c in configs:
    print(c)

# ---------------------------------------------------------------------
# Output reading: n=3,w=2,r=2 -> overlap guaranteed, tolerates 1 failure.
# n=5,w=3,r=3 -> overlap guaranteed, tolerates 2 failures (the "majority" recipe).
# ---------------------------------------------------------------------

In [ ]:
# =====================================================================
# 2. Replication lag backlog
# If a follower is offline for 5 minutes while the leader serves 10k writes/s,
# how much does it need to catch up on when it reconnects?
# =====================================================================

leader_write_qps   = 10_000          # writes per second on leader
outage_seconds     = 5 * 60          # 5-minute follower outage
bytes_per_write    = 1_024           # ~1 KB per write record in the log

backlog_writes = leader_write_qps * outage_seconds
backlog_bytes  = backlog_writes * bytes_per_write
backlog_gib    = backlog_bytes / (1024 ** 3)

print(f"Writes to replay:          {backlog_writes:>12,}")
print(f"Backlog size:              {backlog_bytes:>12,} bytes  (~{backlog_gib:.2f} GiB)")

# Re-streaming that backlog at line rate still consumes the follower's ingest
# bandwidth; the leader also has to keep the log until the follower acks it.
# -> this is the "run out of disk on the leader" risk the chapter warns about.

In [ ]:
# =====================================================================
# 3. Async cross-region replication: lost-write window
# If a leader fails, asynchronously replicated writes younger than one RTT
# may never have reached any follower.
# =====================================================================

rtt_ms           = 150                # typical cross-region RTT (e.g. us-east <-> eu-west)
one_way_ms       = rtt_ms / 2
leader_write_qps = 10_000

# Worst case: a write committed just before the crash takes one-way latency
# to reach the nearest remote follower. Everything written in that window
# is potentially lost on failover.
lost_window_s    = one_way_ms / 1000
max_lost_writes  = int(leader_write_qps * lost_window_s)

print(f"One-way latency:           {one_way_ms:>7.1f} ms")
print(f"Potential lost-write gap:  {lost_window_s*1000:>7.1f} ms")
print(f"Max writes lost on crash:  {max_lost_writes:>7,}")

# ---------------------------------------------------------------------
# Headline: at 10k wps, a 150 ms cross-region RTT exposes up to ~750
# acknowledged writes on a leader crash. Sync replication to one remote
# follower closes this gap but couples write latency to the slowest link.
# ---------------------------------------------------------------------

In [ ]:
# =====================================================================
# 4. Read-your-writes mitigation cost
# If every user read within 60 s of a write goes to the leader, what
# fraction of reads bypasses the follower read pool?
# =====================================================================

reads_per_user_per_hour   = 40
writes_per_user_per_hour  = 2
post_write_leader_window  = 60      # seconds — force-to-leader after a write

# Each write causes up to (post_write_leader_window / 3600) * reads_per_hour
# reads to be steered to the leader. Cap at total reads.
seconds_per_hour = 3600
leader_reads_per_user_per_hour = min(
    reads_per_user_per_hour,
    writes_per_user_per_hour * post_write_leader_window * (reads_per_user_per_hour / seconds_per_hour),
)
leader_read_fraction = leader_reads_per_user_per_hour / reads_per_user_per_hour

print(f"Reads steered to leader:   {leader_read_fraction*100:>5.2f}% of total reads")
# A few percent goes to the leader -> follower read scaling is mostly preserved.

## High-Level Design

```mermaid
graph LR
  C[Client] -->|write| L[(Leader)]
  L ==sync==> F1[(Follower 1)]
  L -.async.-> F2[(Follower 2)]
  L -.async.-> F3[(Follower 3)]
  C -.read.-> F1
  C -.read.-> F2
  C -.read.-> F3
  C -.read-your-write.-> L
```

Writes are serialized by the leader, which streams a **replication log** to every follower. One follower is synchronous (durability on at least two nodes); the rest are asynchronous (throughput, geographical spread). Reads fan out to any replica — except reads that need read-your-writes consistency, which are pinned to the leader or a sync-updated follower.

## Deep Dive 1: Single-Leader Replication

### Sync vs async timeline

```mermaid
sequenceDiagram
    participant Client
    participant Leader
    participant SyncF as Sync Follower
    participant AsyncF as Async Follower
    Client->>Leader: write(x=B)
    Leader->>Leader: write to local WAL
    Leader->>SyncF: replicate(x=B)
    SyncF-->>Leader: ack
    Leader->>AsyncF: replicate(x=B) (fire-and-forget)
    Leader-->>Client: ok (durable on leader + sync follower)
    Note over AsyncF: may arrive seconds or<br/>minutes later under load
    AsyncF-->>Leader: ack (eventually)
```

### Failover hazards

- **Lost writes on async replication.** If the old leader acknowledged writes that had not yet reached the new leader, those writes are commonly *silently discarded* when the old leader rejoins as a follower.
- **External systems diverge.** GitHub 2012: a stale MySQL follower was promoted; its autoincrement counter reused primary keys, and a Redis store keyed by those IDs served one user's data to another.
- **Split brain.** Two nodes both believe they are the leader; both accept writes; data is lost or corrupted. Naive "shoot the other node" logic can end up shutting down *both*.
- **Timeout tuning.** Too short — spurious failovers make an overloaded system worse. Too long — longer user-visible outage on a real leader failure. 30 s is a common starting point.
- **Pick the most up-to-date follower** as the new leader. With sync/semisync that's the one the old leader waited for; with async, the one with the highest log-sequence number.

## Deep Dive 2: Replication Log Formats

| Format | Compactness | Determinism hazard | Coupled to storage engine? | Zero-downtime upgrade? | CDC-friendly? |
| --- | --- | --- | --- | --- | --- |
| **Statement-based** (MySQL <5.1, VoltDB) | Very high — just the SQL | High: `NOW()`, `RAND()`, triggers, autoincrement order | No | Yes (logically decoupled) | Awkward (need to re-execute) |
| **WAL shipping** (PostgreSQL, Oracle) | Low — byte-level block changes | None (bytes are what they are) | Very tightly coupled | **No** — version skew breaks the log | No (physical, not logical) |
| **Logical / row-based** (MySQL binlog, PG logical decoding) | Medium — row-level before/after images | None | No (decoded from physical WAL) | Yes | Yes — the native format for CDC |

The punchline: **WAL shipping is fast and bulletproof but pins leader and follower to the exact same storage-engine version.** Logical logs cost some CPU to decode, but enable rolling upgrades and downstream consumers (search indexes, data warehouses).

## Deep Dive 3: Replication Lag & Consistency Guarantees

Three anomalies fall out of asynchronous followers:

```mermaid
sequenceDiagram
    participant U as User
    participant L as Leader
    participant F1 as Fast Follower
    participant F2 as Lagging Follower
    rect rgb(245, 230, 230)
    Note over U,F2: 1. Read-your-writes violation
    U->>L: write profile
    L-->>U: ok
    U->>F2: read profile
    F2-->>U: (empty — write not yet replicated)
    end
    rect rgb(230, 245, 230)
    Note over U,F2: 2. Monotonic-reads violation
    U->>F1: read comments -> [c1]
    U->>F2: read comments -> []  (time went backward)
    end
    rect rgb(230, 230, 245)
    Note over U,F2: 3. Consistent-prefix violation
    L->>F1: "answer: 10 seconds" (fast)
    L->>F2: "question: how far?" (slow)
    U->>F1: read -> answer arrives before question
    end
```

**Mitigations:**

- **Read-your-writes**: read the user's own records from the leader, or route reads within N seconds of the user's last write to the leader, or attach the write's log-sequence number to the session and block reads on a follower that hasn't caught up.
- **Monotonic reads**: pin each user to one replica (by user-id hash). Simple; fails over awkwardly.
- **Consistent prefix**: keep causally-related writes on the same shard, or track causality explicitly with version vectors and buffer out-of-order deliveries.

## Deep Dive 4: Multi-Leader & Multi-Region

```mermaid
graph TB
  subgraph us-east
    L1[(Leader US)]
    F1a[(Follower)]
    F1b[(Follower)]
    L1 --> F1a
    L1 --> F1b
  end
  subgraph eu-west
    L2[(Leader EU)]
    F2a[(Follower)]
    L2 --> F2a
  end
  subgraph ap-south
    L3[(Leader ASIA)]
    F3a[(Follower)]
    L3 --> F3a
  end
  L1 <-.async cross-region.-> L2
  L2 <-.async cross-region.-> L3
  L1 <-.async cross-region.-> L3
  ClientUS((US client)) --> L1
  ClientEU((EU client)) --> L2
  ClientAS((ASIA client)) --> L3
```

| Dimension | Single-leader across regions | Multi-leader across regions |
| --- | --- | --- |
| Write latency | High — every write pays a cross-region RTT | Low — write to local region, replicate in the background |
| Tolerance of regional outage | Failover to another region (minutes of disruption) | Each region keeps serving; catches up on reconnect |
| Tolerance of inter-region network problems | Very poor — blocks writes from remote regions | High — regions continue independently |
| Achievable consistency | Strong (serializable, linearizable possible) | Weaker — concurrent writes cause conflicts |
| Global constraints (unique usernames, no negative balance) | Easy to enforce | **Cannot** be enforced globally; concurrent leaders can both "succeed" |
| Topologies | N/A | All-to-all (fault-tolerant), star (SPOF on root), circular (SPOF on any node) |

Watch out for: autoincrement-ID collisions, trigger side-effects, and the Figure 6-8 ordering problem (an UPDATE arriving at leader 2 before its INSERT — needs version vectors, not wall clocks, to order correctly).

## Deep Dive 5: Conflict Resolution

When two leaders (or two leaderless replicas) accept writes concurrently, replicas must **converge**. Four strategies, roughly in order of increasing sophistication:

- **Conflict avoidance** — route all writes for a record to the same leader (e.g. a user's home region). Breaks down when the designated leader changes or the user moves.
- **Last-Write-Wins (LWW)** — attach a timestamp, highest wins, others are silently discarded. *Eventually consistent at the cost of data loss.* Highly sensitive to clock skew if wall-clock timestamps are used; logical clocks help but don't fully rescue it.
- **Siblings / manual merge** (CouchDB-style) — keep all concurrent values; return the set on reads; application or user merges. Guarantees no data loss, but the API becomes a set-of-values and app code must merge correctly (Amazon's "deleted cart items reappear" bug).
- **Automatic merge: CRDTs and OT** — algorithmic convergence that *preserves user intent*. **OT** (Google Docs) transforms operation indices; **CRDTs** (Automerge, Yjs, Riak, Azure Cosmos) give each item an immutable ID so merges are pure set operations. Both yield *strong eventual consistency*.

| Strategy | Data loss? | Metadata cost | Can enforce global constraints? | Typical use |
| --- | --- | --- | --- | --- |
| Conflict avoidance | None (when respected) | None | Yes, per-record | Per-user partitioning |
| LWW | Yes, silently | Timestamp per value | No | Cassandra, ScyllaDB |
| Siblings / manual | None | Version vector + N values | No | CouchDB, Riak |
| CRDT / OT | None | Per-character IDs, causal context | No (e.g. "max 5 items") | Google Docs (OT), Figma/Linear/Yjs (CRDT) |

## Deep Dive 6: Leaderless (Dynamo-Style) & Quorums

```mermaid
graph TB
  Client((Client))
  Client -->|write v7 to all| R1[(Replica 1)]
  Client -->|write v7 to all| R2[(Replica 2)]
  Client -->|write v7 to all| R3[(Replica 3 - down)]
  R1 -->|ack| Client
  R2 -->|ack| Client
  Note1[w=2 acks -> write succeeds]
  Client -->|read from all| R1
  Client -->|read from all| R2
  Client -->|read from all| R3b[(Replica 3 - back)]
  R1 -->|v7| Client
  R2 -->|v7| Client
  R3b -->|v6 stale| Client
  Note2[r=2 -> max version v7 wins;<br/>read-repair writes v7 back to R3]
```

With `n=3, w=2, r=2`, `w+r > n` guarantees the read and write sets overlap on at least one replica — so reads are never stuck on a purely stale subset.

**Mechanisms that fill gaps between writes and reads:**

- **Read repair** — when a client observes a stale replica during a parallel read, it writes the fresh value back to that replica. Great for hot keys; useless for keys that are never read.
- **Hinted handoff** — if a replica is down, a peer stores writes "on its behalf" and forwards them when it returns. Generates extra load at exactly the wrong time.
- **Anti-entropy** — background scan that diffs replicas and copies missing data. Slow, unordered, but eventually covers cold keys too.
- **Sloppy quorum** (Riak, Dynamo; "consistency level ANY" in Cassandra) — under partition, accept writes on *any* reachable `w` nodes, even ones outside the key's normal `n`. Keeps writes flowing but drops the `w+r > n` guarantee.

**Edge cases where `w+r > n` still doesn't guarantee freshness:** a failed node restored from an old replica; rebalancing; a write concurrent with a read; a write that succeeded on fewer than `w` replicas but wasn't rolled back on those that did accept it; clock-skew LWW silently dropping writes.

**Why leaderless wins on tail latency**: a leader-based system has to detect failure and fail over, which itself causes a spike. A leaderless system just returns whichever `r` replicas respond first (**request hedging**) and moves on — gray failures and overloaded nodes barely register.

## Deep Dive 7: Concurrent-Write Detection

On a single replica, a per-key **version number** suffices: the client reads, gets the current version plus all siblings, writes back with that version number, and the server overwrites values ≤ that version while keeping values > that version as concurrent siblings.

```mermaid
sequenceDiagram
    participant C1 as Client 1
    participant S as Server
    participant C2 as Client 2
    C1->>S: add(milk)
    S-->>C1: v1 = [milk]
    C2->>S: add(eggs)  (unaware of v1)
    S-->>C2: v2 = {[milk], [eggs]}
    C1->>S: write([milk, flour], based on v1)
    S-->>C1: v3 = {[milk, flour], [eggs]}
    C2->>S: write([eggs, milk, ham], based on v2)
    S-->>C2: v4 = {[milk, flour], [eggs, milk, ham]}
    C1->>S: write([milk, flour, eggs, bacon], based on v3)
    S-->>C1: v5 = {[milk, flour, eggs, bacon], [eggs, milk, ham]}
    Note over S: two concurrent siblings persist<br/>no write was lost
```

With **multiple replicas accepting writes**, a single counter per key is insufficient — different replicas increment independently. The fix is a **version vector**: one counter per `(key, replica)` pair. Replicas compare vectors to decide "A happened-before B" vs "A is concurrent with B"; concurrent values are kept as siblings and replicas that fall behind are caught up.

| Variant | What it counts | Distinguishes concurrent from causal? | Where it's used |
| --- | --- | --- | --- |
| **Version number** | Per key, one counter | Yes, on one replica only | Single-replica servers, shopping-cart example |
| **Version vector** | Per key, per replica | Yes, across replicas | Generic leaderless replication |
| **Dotted version vector** | Per key, per replica, with "dots" for individual writes | Yes, with lower metadata overhead | Riak 2.0 |
| **Vector clock** | Per process, not per key | Yes (for events, not data) | Lamport-style system modeling |

**Why wall-clock timestamps are insufficient**: two clocks that are out of sync by even milliseconds can re-order causally-related writes. LWW-based systems can silently drop the "later" write because another node with a fast clock stamped an earlier value with a higher timestamp. Version vectors route around clocks entirely and rely on observed causality.

## Trade-offs and Alternatives

| Dimension | Single-leader | Multi-leader | Leaderless (Dynamo-style) |
| --- | --- | --- | --- |
| Write path | All writes to one node | Writes to any leader, async cross-leader | Writes to `w` of `n` replicas in parallel |
| Strongest consistency achievable | Linearizable (with sync) | Eventual + per-user guarantees | Eventual; `w+r>n` probabilistic freshness |
| Conflict handling | None needed | Required (LWW, CRDT, manual) | Required (LWW, CRDT, manual) |
| Failover behavior | Explicit leader election, risk of split brain and lost writes | Each leader independent; no failover | **No failover** — replicas are peers |
| Geo-distribution fit | Poor — every write crosses regions | Excellent — one leader per region | Excellent — tunable per-region quorums |
| Tail-latency resilience | Poor — leader is a bottleneck; gray failures hurt | Good | Best — request hedging, no quorum-of-one failure |
| Constraint enforcement (uniqueness, balance ≥ 0) | Yes | **No** — concurrent leaders break invariants | **No** |
| Operational complexity | Low — familiar model | High — conflict resolution + topology | Medium — quorum tuning, anti-entropy, hints |
| Typical systems | PostgreSQL, MySQL, MongoDB, Kafka, DynamoDB | MySQL bidirectional, YugabyteDB, sync engines, Google Docs | Cassandra, ScyllaDB, Riak, original Dynamo |

## Key Takeaways

1. **Replication is about change under uncertainty.** Copying static data is trivial; the entire discipline exists because networks, clocks, and nodes fail.
2. **There are exactly three models.** Single-leader, multi-leader, leaderless. Everything else (consensus, NewSQL, sync engines, Dynamo) is a refinement of one of them.
3. **Sync vs async is a trade-off, not a setting you pick once.** Semisynchronous (one sync follower, rest async) is the practical default — durability on two nodes without stalling on a slow third.
4. **Failover has no easy answers.** Async replication plus fast failover can silently destroy acknowledged writes. Split-brain fencing is fraught. Many ops teams prefer manual failover despite the downtime.
5. **Replication lag creates user-visible anomalies** — read-your-writes, monotonic reads, consistent prefix — that the database can't solve for you unless it gives you linearizability. Most systems push these up into app code or session stickiness.
6. **Multi-leader buys latency and regional resilience at the cost of global constraints.** No multi-leader system can enforce "username is unique" or "balance never negative" without sacrificing multi-leader-ness.
7. **Quorums (`w+r>n`) are probabilistic, not ironclad.** Edge cases — failed-node restore, rebalancing, concurrent read/write, partial writes, clock-skew LWW — can break freshness even when the arithmetic holds.
8. **Wall-clock timestamps can't order causally-related writes.** Version vectors (and dotted version vectors, vector clocks, CRDTs) do the real work of distinguishing "A happened-before B" from "A and B are concurrent" — the prerequisite for any conflict resolution that doesn't silently drop data.

## See Also

- [[01-single-leader-replication-and-logs]]
- [[02-replication-lag-and-consistency-guarantees]]
- [[03-multi-leader-replication-and-topologies]]
- [[04-sync-engines-and-local-first-software]]
- [[05-conflict-resolution-lww-crdts-ot]]
- [[06-leaderless-replication-and-quorums]]
- [[07-detecting-concurrent-writes-version-vectors]]